### Setup and Configuration
Import libraries, locate the project root from any notebook working directory, and define the raw dataset/config paths.


In [ ]:
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
from IPython.display import display


def find_project_root(start: Path) -> Path:
    """Find the repository root from the current notebook working directory."""
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "config.yaml").exists() and (candidate / "data" / "raw").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root. Run this notebook inside the project directory.")


PROJECT_ROOT = find_project_root(Path.cwd())
CONFIG_PATH = PROJECT_ROOT / "configs" / "config.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "CIC-ToN-IoT.csv"
SAMPLE_SIZE = 100_000

if not CONFIG_PATH.exists():
    raise FileNotFoundError(f"Config file not found: {CONFIG_PATH}")
if not DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset file not found: {DATA_PATH}")

print("Project root:", PROJECT_ROOT)
print("Config path:", CONFIG_PATH)
print("Dataset path:", DATA_PATH)


### Load Configuration
Load expected dataset metadata, selected feature columns, drop columns, and label settings from `configs/config.yaml`.


In [ ]:
with open(CONFIG_PATH, "r", encoding="utf-8") as file:
    config = yaml.safe_load(file)

data_config = config["data"]
label_column = data_config["label_column"]
feature_columns = data_config["feature_columns"]
drop_columns = data_config["drop_columns"]

print("Expected rows:", data_config["expected_rows"])
print("Expected total columns:", data_config["expected_total_columns"])
print("Expected feature count:", data_config["expected_feature_count"])
print("Configured label column:", label_column)
print("Configured feature columns:", len(feature_columns))
print("Configured drop columns:", len(drop_columns))


### Read Header and Sample
Read only the CSV header and a sample so this notebook remains fast and safe on a 3 GB dataset.


In [ ]:
header_columns = pd.read_csv(DATA_PATH, nrows=0).columns.str.strip().tolist()
sample_df = pd.read_csv(DATA_PATH, nrows=SAMPLE_SIZE, low_memory=False)
sample_df.columns = sample_df.columns.str.strip()

print("Header columns:", len(header_columns))
print("Sample shape:", sample_df.shape)
display(sample_df.head())


### Validate Dataset Structure
Check that the raw dataset structure matches the proposal and the project configuration.


In [ ]:
missing_config_features = sorted(set(feature_columns) - set(header_columns))
missing_drop_columns = sorted(set(drop_columns) - set(header_columns))
unknown_columns = sorted(set(header_columns) - set(feature_columns) - set(drop_columns) - {label_column})
feature_drop_overlap = sorted(set(feature_columns) & set(drop_columns))

print("Header column count matches config:", len(header_columns) == data_config["expected_total_columns"])
print("Feature count matches config:", len(feature_columns) == data_config["expected_feature_count"])
print("Missing configured features:", missing_config_features)
print("Missing configured drop columns:", missing_drop_columns)
print("Feature/drop overlap:", feature_drop_overlap)
print("Unmapped raw columns:", unknown_columns)

if label_column not in header_columns:
    raise ValueError(f"Label column '{label_column}' was not found in the raw dataset.")
if missing_config_features or missing_drop_columns or feature_drop_overlap or unknown_columns:
    raise ValueError("Dataset column mapping does not match config.yaml. Review the printed lists above.")


### Inspect Sample Class Distribution
Display class counts in the sample as a quick sanity check before running the full chunked profile in notebook 01.


In [ ]:
sample_class_distribution = (
    sample_df[label_column]
    .value_counts(dropna=False)
    .rename_axis("class")
    .reset_index(name="sample_count")
)
sample_class_distribution["sample_percentage"] = (
    sample_class_distribution["sample_count"] / sample_class_distribution["sample_count"].sum() * 100
)

display(sample_class_distribution)
print("Sample classes:", sample_class_distribution.shape[0])


### Build Sample Variation Summary
Compute sample-level uniqueness, zero ratio, and standard deviation for the configured 67 model features.


In [ ]:
X_sample = sample_df[feature_columns].apply(pd.to_numeric, errors="coerce")
X_sample = X_sample.replace([np.inf, -np.inf], np.nan)

variation_summary = pd.DataFrame({
    "feature": feature_columns,
    "sample_missing_count": X_sample.isna().sum().values,
    "sample_missing_percentage": (X_sample.isna().mean() * 100).values,
    "sample_unique_count": X_sample.nunique(dropna=False).values,
    "sample_zero_ratio": (X_sample == 0).mean().values,
    "sample_std": X_sample.std(numeric_only=True).values,
})

variation_summary = variation_summary.sort_values(
    by=["sample_unique_count", "sample_zero_ratio", "sample_std"],
    ascending=[True, False, True],
)

display(variation_summary.head(20))


### Inspect Configured Drop Columns
Review the columns excluded from modeling and summarize their sample-level variation where numeric values are available.


In [ ]:
drop_sample = sample_df[drop_columns].copy()
drop_summary_rows = []

for column in drop_columns:
    series = drop_sample[column]
    numeric_series = pd.to_numeric(series, errors="coerce")
    drop_summary_rows.append({
        "column": column,
        "sample_dtype": str(series.dtype),
        "sample_unique_count": series.nunique(dropna=False),
        "numeric_non_null_ratio": numeric_series.notna().mean(),
        "sample_zero_ratio": (numeric_series == 0).mean() if numeric_series.notna().any() else np.nan,
    })

drop_summary = pd.DataFrame(drop_summary_rows).sort_values(
    by=["numeric_non_null_ratio", "sample_unique_count"],
    ascending=[True, True],
)

display(drop_summary)


### Final Notebook Sanity Checks
Assert the key Phase 1 assumptions that should hold before moving into preprocessing.


In [ ]:
assert len(header_columns) == data_config["expected_total_columns"]
assert len(feature_columns) == data_config["expected_feature_count"]
assert label_column in sample_df.columns
assert set(feature_columns).isdisjoint(drop_columns)
assert len(set(feature_columns) | set(drop_columns) | {label_column}) == len(header_columns)

print("Dataset variation notebook checks passed.")
print("Configured model features:", len(feature_columns))
print("Excluded non-model columns:", len(drop_columns))
print("Label column:", label_column)
